In [61]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
import os, getpass

from langchain.globals import set_debug, set_verbose
set_debug(False)
set_verbose(False)

# # GPT-5-Mini
# diagnose_llm = ChatOpenAI(
#     model="gpt-5-mini",
#     reasoning={"effort":"low"},
#     temperature=0.3,
#     max_completion_tokens=1600,
#     use_responses_api=True
# )

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")
    
# Gemini-2.5-Flash
diagnose_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    max_output_tokens = 1800,
    thinking_budget = 1000,
    temperature = 0.3,
    top_p = 0.95
)

E0000 00:00:1759876205.476528 11616990 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [ ]:
DX_PROMPT = """
You are an expert AI Dermatologist. Your task is to examine the dermatoscopic image and patient metadata to determine the **most likely diagnosis** from the following list:

- Nevus
- Melanoma
- Basal cell carcinoma
- Squamous cell carcinoma
- Pigmented benign keratosis
- Actinic keratosis
- Dermatofibroma

Instructions:
1. Carefully examine the dermatoscopic image.
2. Consider the patient’s metadata (age, sex, anatomical site) to support your reasoning.
3. Identify key visual features: pigment patterns, structures, vascular patterns, surface characteristics, distribution, architecture and other relevant factors.
4. Based on your expert knowledge, determine the single best diagnosis and one differential diagnosis.
5. Clearly justify your decision by referring to the observed distinctive dermoscopic features and their alignment with typical diagnostic patterns.

Output ONLY a single JSON object (No Markdown):
{
  "diagnosis": "<The final diagnosis>",
  "confidence": "<Low | Medium | High>",
  "differential_diagnosis": "<The second most plausible diagnosis>",
  "reasoning": "<A concise but structured explanation of how you arrived at this diagnosis, describing key image findings and how they support your choice>"
}
"""

In [63]:
import pickle as pkl
from pathlib import Path

with open("../dataset/300_test_set.pkl", "rb") as f:
    test_set = pkl.load(f)
    
image_path = "../dataset/test"
for item in test_set:
    file_name = Path(item["image_path"]).name
    item["image_path"] = f"{image_path}/{file_name}"

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableLambda

from utils import encode_image
from typing import List, Dict, Any


def prepare_input(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    base64_image = encode_image(data['image_path'])
    image_data = f"data:image/jpeg;base64,{base64_image}"
    metadata_text = (
        f"--- Patient Metadata ---\n"
        f"Age: {data['age']}\n"
        f"Sex: {data['sex']}\n"
        f"Lesion Location: {data['anatom_site']}\n"
    )
    
    message = [
        { "type": "text", "text": f"Patient Data:\n {metadata_text}\n" },
        { "type": "image_url", "image_url": { "url": f"{image_data}" } }
    ]
    
    messages =[
        SystemMessage(content=DX_PROMPT),
        HumanMessage(content=message)
    ]
    
    return messages

parser = JsonOutputParser()
dx_chain = (
    RunnableLambda(prepare_input)
    | diagnose_llm
    | parser
)


In [72]:
from tqdm import tqdm
from pathlib import Path
import json

# OUT_FILE = "../results/baseline/gpt/gpt-baseline.jsonl"
OUT_FILE = "../results/baseline/gemini/gemini-baseline.jsonl"

print(f"Starting processing and writing results to {OUT_FILE}...")

with open(OUT_FILE, "w") as f:
    for i, item in enumerate(tqdm(test_set, desc="Processing Test Cases")): 
        case_id = Path(item['image_path']).stem
        
        try:
            result = dx_chain.invoke(item)
            
            output_line = {
                "case_id": case_id,
                "image_path": item['image_path'],
                "result": result,
                "ground_truth": item['diagnosis']
            }
            
            json_line = json.dumps(output_line)
            f.write(json_line + '\n')
            
        except Exception as e:
            tqdm.write(f"Error processing {item['image_path']}: {e}")
            
            # Write an error log to the JSONL file
            error_line = json.dumps({
                "case_id": case_id,
                "input_path": item['image_path'],
                "error": str(e)
            })
            f.write(error_line + '\n')

Starting processing and writing results to ../results/baseline/gemini/gemini-baseline.jsonl...


Processing Test Cases:  98%|█████████▊| 295/300 [41:15<00:38,  7.76s/it]Gemini produced an empty response. Continuing with empty message
Feedback: block_reason: OTHER

Processing Test Cases:  99%|█████████▊| 296/300 [41:15<00:22,  5.56s/it]

Error processing ../dataset/test/ISIC_0025213.jpg: Invalid json output: 
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 


Processing Test Cases: 100%|██████████| 300/300 [41:47<00:00,  8.36s/it]
